# Exploración: APIS.NET.PE - Pasarela de Consulta RUC y DNI

- **Tipo de fuente:** Servicio agregador / API REST de terceros (no gubernamental) que sincroniza padrones públicos de SUNAT y RENIEC.
- **Descripción y propósito:** Actúa como fuente de consulta rápida de baja latencia (`< 300 ms`) y mecanismo de contraste o *fallback* frente a las fuentes oficiales. Permite obtener datos básicos de identificación y ubicación fiscal sin requerir navegación en navegador (`Playwright`) ni resolución de CAPTCHAs.
- **Endpoints base:**
  - Consulta RUC: `https://api.apis.net.pe/v1/ruc?numero={ruc}`
  - Consulta DNI: `https://api.apis.net.pe/v1/dni?numero={dni}`
- **Criterios de búsqueda soportados:**
  - **Por RUC (11 dígitos):** Datos de padrón reducidos del contribuyente.
  - **Por DNI (8 dígitos):** Identificación civil básica de personas naturales.
- **Información que se obtiene:**
  - **En consultas por RUC:** Razón Social (`nombre`), tipo y número de documento, estado (`ACTIVO`, `BAJA DE OFICIO`, etc.), condición de domicilio (`HABIDO`, `NO HABIDO`), dirección fiscal completa, código de ubigeo y desglose geográfico (`departamento`, `provincia`, `distrito`, `viaTipo`, `viaNombre`, `zonaTipo`, `numero`, `lote`, etc.).
  - **En consultas por DNI:** Número de documento, nombre completo concatenado (`nombre`), nombres (`nombres`), apellido paterno (`apellidoPaterno`), apellido materno (`apellidoMaterno`) y dígito verificador.

## Estrategia

- **Arquitectura del servicio:** API REST pública sobre HTTPS que responde directamente en formato `application/json` mediante peticiones `GET` simples.
- **Cliente asíncrono ligero (`httpx`):**
  1. No requiere motor de renderizado JS ni evasión de Cloudflare Turnstile. Se utiliza `httpx.AsyncClient` con cabeceras estándar de navegador (`User-Agent`, `Accept: application/json` y `Referer: https://apis.net.pe/`) para evitar bloqueos por *user-agent* de script por defecto.
  2. Se mide el tiempo de respuesta en milisegundos (`duration_ms`) para registrar la telemetría de auditoría de la fuente.
- **Validación y normalización:**
  - Antes de emitir la petición HTTP, se valida en local que el parámetro cumpla estrictamente con la longitud y formato numérico (**11 dígitos para RUC** u **8 dígitos para DNI**).
  - La respuesta JSON cruda (`raw`) se mapea a un esquema estandarizado en inglés (`business_name`, `status`, `condition`, `address`, `ubigeo`, etc.) listo para integrarse con los puertos de salida de la arquitectura hexagonal.

In [1]:
import httpx
import json
from datetime import datetime, timezone
from typing import Dict, Any

# Configuración exclusiva para consulta por RUC (11 dígitos)
RUC_PRUEBA = "20607524476"
URL_APIS_RUC = "https://api.apis.net.pe/v1/ruc"

HEADERS_APIS_NET = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "application/json",
    "Referer": "https://apis.net.pe/"
}

async def consultar_ruc_apisnet(ruc: str) -> Dict[str, Any]:
    ruc_limpio = ruc.strip()
    if len(ruc_limpio) != 11 or not ruc_limpio.isdigit():
        return {"error": "La consulta requiere exclusivamente un RUC numérico de 11 dígitos."}

    inicio = datetime.now(timezone.utc)

    async with httpx.AsyncClient(timeout=15.0, headers=HEADERS_APIS_NET) as client:
        try:
            print(f"[1/2] Consultando RUC {ruc_limpio} en {URL_APIS_RUC}...")
            resp = await client.get(URL_APIS_RUC, params={"numero": ruc_limpio})
            duracion = int((datetime.now(timezone.utc) - inicio).total_seconds() * 1000)

            if resp.status_code == 200:
                raw = resp.json()
                print(f"[2/2] ¡Respuesta exitosa (200 OK) en {duracion} ms!")
                return {
                    "audit": {
                        "source": "APIS_NET_PE_RUC",
                        "tax_id": ruc_limpio,
                        "status_code": 200,
                        "duration_ms": duracion,
                        "success": True
                    },
                    "data": {
                        "document_type": "RUC",
                        "document_number": raw.get("numeroDocumento"),
                        "business_name": raw.get("nombre"),
                        "status": raw.get("estado"),
                        "condition": raw.get("condicion"),
                        "address": raw.get("direccion"),
                        "ubigeo": raw.get("ubigeo"),
                        "department": raw.get("departamento"),
                        "province": raw.get("provincia"),
                        "district": raw.get("distrito"),
                        "via_type": raw.get("viaTipo"),
                        "via_name": raw.get("viaNombre"),
                        "zone_type": raw.get("zonaTipo"),
                        "zone_code": raw.get("zonaCodigo"),
                        "number": raw.get("numero"),
                        "lot": raw.get("lote"),
                        "apartment": raw.get("dpto"),
                        "block": raw.get("manzana"),
                        "kilometer": raw.get("kilometro")
                    },
                    "raw": raw
                }

            print(f"[2/2] La API respondió con estado HTTP {resp.status_code} ({duracion} ms).")
            return {
                "audit": {
                    "source": "APIS_NET_PE_RUC",
                    "tax_id": ruc_limpio,
                    "status_code": resp.status_code,
                    "duration_ms": duracion,
                    "success": False
                },
                "data": None,
                "raw": resp.text[:200]
            }

        except Exception as exc:
            return {
                "audit": {
                    "source": "APIS_NET_PE_RUC",
                    "tax_id": ruc_limpio,
                    "status_code": None,
                    "duration_ms": None,
                    "success": False
                },
                "data": None,
                "error": str(exc)
            }

# Ejecución en el notebook
res_ruc = await consultar_ruc_apisnet(RUC_PRUEBA)
print("\n================== RESULTADO APIS.NET.PE (RUC) ==================")
print(json.dumps(res_ruc, indent=2, ensure_ascii=False))

[1/2] Consultando RUC 20607524476 en https://api.apis.net.pe/v1/ruc...
[2/2] ¡Respuesta exitosa (200 OK) en 632 ms!

================== RESULTADO APIS.NET.PE (RUC) ==================
{
  "audit": {
    "source": "APIS_NET_PE_RUC",
    "tax_id": "20607524476",
    "status_code": 200,
    "duration_ms": 632,
    "success": true
  },
  "data": {
    "document_type": "RUC",
    "document_number": "20607524476",
    "business_name": "INVERSIONES Y SERVICIOS HUAYNA DEL PERU EMPRESA INDIVIDUAL DE RESPONSABILIDAD LIMITADA",
    "status": "BAJA DE OFICIO",
    "condition": "HABIDO",
    "address": "CAL. VILLA LA RINCONADA LOTE H-1 APV. VILLA LA RINCONADA ",
    "ubigeo": "080104",
    "department": "CUSCO",
    "province": "CUSCO",
    "district": "SAN JERONIMO",
    "via_type": "CAL.",
    "via_name": "VILLA LA RINCONADA",
    "zone_type": "VILLA LA RINCONADA",
    "zone_code": "APV.",
    "number": "-",
    "lot": "H-1",
    "apartment": "-",
    "block": "-",
    "kilometer": "-"
  },
  "raw

In [2]:
import httpx
import json
from datetime import datetime, timezone
from typing import Dict, Any

# Configuración exclusiva para consulta por DNI (8 dígitos)
DNI_PRUEBA = "44900952"
URL_APIS_DNI = "https://api.apis.net.pe/v1/dni"

HEADERS_APIS_NET = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "application/json",
    "Referer": "https://apis.net.pe/"
}

async def consultar_dni_apisnet(dni: str) -> Dict[str, Any]:
    dni_limpio = dni.strip()
    if len(dni_limpio) != 8 or not dni_limpio.isdigit():
        return {"error": "La consulta requiere exclusivamente un DNI numérico de 8 dígitos."}

    inicio = datetime.now(timezone.utc)

    async with httpx.AsyncClient(timeout=15.0, headers=HEADERS_APIS_NET) as client:
        try:
            print(f"[1/2] Consultando DNI {dni_limpio} en {URL_APIS_DNI}...")
            resp = await client.get(URL_APIS_DNI, params={"numero": dni_limpio})
            duracion = int((datetime.now(timezone.utc) - inicio).total_seconds() * 1000)

            if resp.status_code == 200:
                raw = resp.json()
                print(f"[2/2] ¡Respuesta exitosa (200 OK) en {duracion} ms!")
                return {
                    "audit": {
                        "source": "APIS_NET_PE_DNI",
                        "tax_id": dni_limpio,
                        "status_code": 200,
                        "duration_ms": duracion,
                        "success": True
                    },
                    "data": {
                        "document_type": "DNI",
                        "document_number": raw.get("numeroDocumento"),
                        "full_name": raw.get("nombre"),
                        "names": raw.get("nombres"),
                        "first_surname": raw.get("apellidoPaterno"),
                        "second_surname": raw.get("apellidoMaterno"),
                        "check_digit": raw.get("digitoVerificador")
                    },
                    "raw": raw
                }

            print(f"[2/2] La API respondió con estado HTTP {resp.status_code} ({duracion} ms).")
            return {
                "audit": {
                    "source": "APIS_NET_PE_DNI",
                    "tax_id": dni_limpio,
                    "status_code": resp.status_code,
                    "duration_ms": duracion,
                    "success": False
                },
                "data": None,
                "raw": resp.text[:200]
            }

        except Exception as exc:
            return {
                "audit": {
                    "source": "APIS_NET_PE_DNI",
                    "tax_id": dni_limpio,
                    "status_code": None,
                    "duration_ms": None,
                    "success": False
                },
                "data": None,
                "error": str(exc)
            }

# Ejecución en el notebook
res_dni = await consultar_dni_apisnet(DNI_PRUEBA)
print("\n================== RESULTADO APIS.NET.PE (DNI) ==================")
print(json.dumps(res_dni, indent=2, ensure_ascii=False))

[1/2] Consultando DNI 44900952 en https://api.apis.net.pe/v1/dni...
[2/2] ¡Respuesta exitosa (200 OK) en 505 ms!

================== RESULTADO APIS.NET.PE (DNI) ==================
{
  "audit": {
    "source": "APIS_NET_PE_DNI",
    "tax_id": "44900952",
    "status_code": 200,
    "duration_ms": 505,
    "success": true
  },
  "data": {
    "document_type": "DNI",
    "document_number": "44900952",
    "full_name": "RONALD RUIZ AUCCAPIÑA",
    "names": "RONALD",
    "first_surname": "RUIZ",
    "second_surname": "AUCCAPIÑA",
    "check_digit": null
  },
  "raw": {
    "nombre": "RONALD RUIZ AUCCAPIÑA",
    "tipoDocumento": "1",
    "numeroDocumento": "44900952",
    "estado": "",
    "condicion": "",
    "direccion": "",
    "ubigeo": "",
    "viaTipo": "",
    "viaNombre": "",
    "zonaCodigo": "",
    "zonaTipo": "",
    "numero": "",
    "interior": "",
    "lote": "",
    "dpto": "",
    "manzana": "",
    "kilometro": "",
    "distrito": "",
    "provincia": "",
    "departamento"